<a href="https://colab.research.google.com/github/rayana11-blip/analisis_penggunaan_tiktok-instagram/blob/main/Analisis_TikTok-Instagram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# 1. Import Library

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from google.colab import files
from IPython.display import display

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


In [19]:
# 2. Upload Dataset

uploaded = files.upload()

# Mengambil nama file yang diunggah
filename = next(iter(uploaded))

# Membaca dataset
df_raw = pd.read_csv(filename)

print("Nama file:", filename)
print("Dataset berhasil dibaca.")

Saving Analisis Perbandingan Aplikasi Media Sosial Tiktok dan Instagram pada Mahasiswa_i Fakultas Vokasi Universitas Sumatera Utara (Responses) - Form Responses 1.csv to Analisis Perbandingan Aplikasi Media Sosial Tiktok dan Instagram pada Mahasiswa_i Fakultas Vokasi Universitas Sumatera Utara (Responses) - Form Responses 1 (1).csv
Nama file: Analisis Perbandingan Aplikasi Media Sosial Tiktok dan Instagram pada Mahasiswa_i Fakultas Vokasi Universitas Sumatera Utara (Responses) - Form Responses 1 (1).csv
Dataset berhasil dibaca.


In [20]:
# 3. Checking Struktur Dataset

print("Jumlah baris:", df_raw.shape[0])
print("Jumlah kolom:", df_raw.shape[1])

print("\nNama Kolom Dataset:")

for i, col in enumerate(df_raw.columns, start=1):
    print(i, repr(col))

print("\nLima Data Pertama:")
display(df_raw.head())

Jumlah baris: 100
Jumlah kolom: 11

Nama Kolom Dataset:
1 'Timestamp'
2 'ID'
3 'Gender'
4 'Major'
5 'Year'
6 'Tiktok Status'
7 'Tiktok Duration'
8 'Tiktok Purpose'
9 'Instagram Status'
10 'Instagram Duration'
11 'Instagram Purpose'

Lima Data Pertama:


,Timestamp,ID,Gender,Major,Year,Tiktok Status,Tiktok Duration,Tiktok Purpose,Instagram Status,Instagram Duration,Instagram Purpose
0,5/13/2026 13:41,ID001,Perempuan,Statistika,2025,Ya,2 jam 3 menit,Hiburan,Ya,1 jam 3 menit,Sosial
1,5/13/2026 13:41,ID002,Perempuan,Statistika,2025,Ya,2 jam 4 menit,Hiburan,Ya,17 menit,Hiburan
2,5/13/2026 13:43,ID003,Perempuan,Statistika,2025,Ya,1 jam 40 menit,Hiburan,Ya,2 jam,Sosial
3,5/13/2026 13:46,ID004,Perempuan,Statistika,2025,Ya,1 jam,Hiburan,Ya,sepuluh menit,Sosial
4,5/13/2026 13:46,ID005,Perempuan,Statistika,2025,Ya,2 jam 39 menit,Hiburan,Ya,29 menit,Hiburan


In [21]:
# 4. Menyesuaikan Nama Kolom

df = df_raw.copy()

# Membersihkan spasi pada nama kolom
df.columns = df.columns.astype(str).str.strip()

# Mengubah nama kolom tertentu
rename_map = {
    'Tiktok Status': 'TikTok_Status',
    'Tiktok Duration': 'TikTok_Duration',
    'Tiktok Purpose': 'TikTok_Purpose',
    'Instagram Status': 'Instagram_Status',
    'Instagram Duration': 'Instagram_Duration',
    'Instagram Purpose': 'Instagram_Purpose'
}

df = df.rename(columns=rename_map)

# Memeriksa kolom yang diperlukan
required_cols = [
    'ID',
    'Gender',
    'Major',
    'Year',
    'TikTok_Status',
    'TikTok_Duration',
    'TikTok_Purpose',
    'Instagram_Status',
    'Instagram_Duration',
    'Instagram_Purpose'
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Kolom tidak ditemukan: {missing_cols}"
    )

print("Nama kolom berhasil disesuaikan.")
print(df.columns.tolist())

Nama kolom berhasil disesuaikan.
['Timestamp', 'ID', 'Gender', 'Major', 'Year', 'TikTok_Status', 'TikTok_Duration', 'TikTok_Purpose', 'Instagram_Status', 'Instagram_Duration', 'Instagram_Purpose']


In [22]:
# 5. Membuat ID Anonim

# Mempertahankan ID asli
df = df.rename(
    columns={'ID': 'Original_ID'}
)

# Menghapus timestamp karena tidak diperlukan
df = df.drop(
    columns=['Timestamp'],
    errors='ignore'
)

# Membuat ID baru
df = df.reset_index(drop=True)

df.insert(
    0,
    'id',
    range(1, len(df) + 1)
)

print("ID anonim berhasil dibuat.")

display(df.head())

ID anonim berhasil dibuat.


,id,Original_ID,Gender,Major,Year,TikTok_Status,TikTok_Duration,TikTok_Purpose,Instagram_Status,Instagram_Duration,Instagram_Purpose
0,1,ID001,Perempuan,Statistika,2025,Ya,2 jam 3 menit,Hiburan,Ya,1 jam 3 menit,Sosial
1,2,ID002,Perempuan,Statistika,2025,Ya,2 jam 4 menit,Hiburan,Ya,17 menit,Hiburan
2,3,ID003,Perempuan,Statistika,2025,Ya,1 jam 40 menit,Hiburan,Ya,2 jam,Sosial
3,4,ID004,Perempuan,Statistika,2025,Ya,1 jam,Hiburan,Ya,sepuluh menit,Sosial
4,5,ID005,Perempuan,Statistika,2025,Ya,2 jam 39 menit,Hiburan,Ya,29 menit,Hiburan


In [23]:
# 6. Checking Missing Value

missing_check = pd.DataFrame({
    'Jumlah Missing':
        df.isnull().sum(),

    'Persentase Missing':
        (df.isnull().mean() * 100).round(2)
})

display(missing_check)

print(
    "Total seluruh data kosong:",
    df.isnull().sum().sum()
)

,Jumlah Missing,Persentase Missing
id,0,0.0
Original_ID,0,0.0
Gender,0,0.0
Major,0,0.0
Year,0,0.0
TikTok_Status,0,0.0
TikTok_Duration,16,16.0
TikTok_Purpose,16,16.0
Instagram_Status,0,0.0
Instagram_Duration,3,3.0


Total seluruh data kosong: 38


In [24]:
# 7. Checking Duplicate Data

print(
    "Duplikasi seluruh baris:",
    df.duplicated().sum()
)

print(
    "Duplikasi Original_ID:",
    df['Original_ID'].duplicated().sum()
)

# Menampilkan data yang terduplikasi
duplicate_data = df[
    df.duplicated(keep=False)
]

display(duplicate_data)

Duplikasi seluruh baris: 0
Duplikasi Original_ID: 0


,id,Original_ID,Gender,Major,Year,TikTok_Status,TikTok_Duration,TikTok_Purpose,Instagram_Status,Instagram_Duration,Instagram_Purpose


In [25]:
# 8. Cleaning Data Teks

text_cols = [
    'Gender',
    'Major',
    'TikTok_Status',
    'TikTok_Duration',
    'TikTok_Purpose',
    'Instagram_Status',
    'Instagram_Duration',
    'Instagram_Purpose'
]

# Membersihkan spasi
for col in text_cols:
    df[col] = (
        df[col]
        .astype('string')
        .str.strip()
    )

print("Cleaning teks berhasil.")

display(df.head())

Cleaning teks berhasil.


,id,Original_ID,Gender,Major,Year,TikTok_Status,TikTok_Duration,TikTok_Purpose,Instagram_Status,Instagram_Duration,Instagram_Purpose
0,1,ID001,Perempuan,Statistika,2025,Ya,2 jam 3 menit,Hiburan,Ya,1 jam 3 menit,Sosial
1,2,ID002,Perempuan,Statistika,2025,Ya,2 jam 4 menit,Hiburan,Ya,17 menit,Hiburan
2,3,ID003,Perempuan,Statistika,2025,Ya,1 jam 40 menit,Hiburan,Ya,2 jam,Sosial
3,4,ID004,Perempuan,Statistika,2025,Ya,1 jam,Hiburan,Ya,sepuluh menit,Sosial
4,5,ID005,Perempuan,Statistika,2025,Ya,2 jam 39 menit,Hiburan,Ya,29 menit,Hiburan


In [26]:
# 9. Normalisasi Status

def normalize_status(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if value in ['ya', 'yes', 'iya']:
        return 'Ya'

    elif value in ['tidak', 'no']:
        return 'Tidak'

    else:
        return str(value).strip().capitalize()


df['TikTok_Status'] = (
    df['TikTok_Status']
    .apply(normalize_status)
)

df['Instagram_Status'] = (
    df['Instagram_Status']
    .apply(normalize_status)
)

print("Normalisasi status berhasil.")

print("\nStatus TikTok:")
display(df['TikTok_Status'].value_counts(dropna=False))

print("\nStatus Instagram:")
display(df['Instagram_Status'].value_counts(dropna=False))

Normalisasi status berhasil.

Status TikTok:


,count
TikTok_Status,
Ya,84
Tidak,16



Status Instagram:


,count
Instagram_Status,
Ya,97
Tidak,3


In [27]:
# 10. Fungsi Konversi Durasi

def duration_to_minutes(value):

    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    if text == '':
        return np.nan

    # Memperbaiki variasi penulisan
    text = text.replace('mebit', 'menit')
    text = text.replace('mnt', 'menit')
    text = text.replace('mins', 'menit')

    # Menghilangkan keterangan perkiraan
    text = text.replace('kurang lebih', '')
    text = text.replace('sekitar', '')

    # Menangani kata bilangan
    if text == 'sepuluh menit':
        return 10

    # Merapikan penulisan angka dan satuan
    text = re.sub(r'(?<=\d)jam', ' jam', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Menangani rentang, misalnya 1-2 jam
    range_match = re.search(
        r'(\d+(?:[.,]\d+)?)\s*-\s*'
        r'(\d+(?:[.,]\d+)?)\s*'
        r'(jam|menit|detik)',
        text
    )

    if range_match:

        angka1 = float(
            range_match.group(1).replace(',', '.')
        )

        angka2 = float(
            range_match.group(2).replace(',', '.')
        )

        satuan = range_match.group(3)

        nilai_tengah = (angka1 + angka2) / 2

        if satuan == 'jam':
            return nilai_tengah * 60

        elif satuan == 'menit':
            return nilai_tengah

        elif satuan == 'detik':
            return nilai_tengah / 60

    # Menghitung jam, menit, dan detik
    total = 0
    ditemukan = False

    pola = [
        (r'(\d+(?:[.,]\d+)?)\s*jam', 60),
        (r'(\d+(?:[.,]\d+)?)\s*menit', 1),
        (r'(\d+(?:[.,]\d+)?)\s*detik', 1/60)
    ]

    for pattern, multiplier in pola:

        match = re.search(pattern, text)

        if match:

            angka = float(
                match.group(1).replace(',', '.')
            )

            total += angka * multiplier

            ditemukan = True

    if ditemukan:
        return total

    return np.nan

In [28]:
# 11. Konversi Durasi TikTok dan Instagram

df['TikTok_Minutes'] = (
    df['TikTok_Duration']
    .apply(duration_to_minutes)
)

df['Instagram_Minutes'] = (
    df['Instagram_Duration']
    .apply(duration_to_minutes)
)

print("Konversi durasi berhasil.")

display(
    df[[
        'id',
        'TikTok_Duration',
        'TikTok_Minutes',
        'Instagram_Duration',
        'Instagram_Minutes'
    ]].head(15)
)

Konversi durasi berhasil.


,id,TikTok_Duration,TikTok_Minutes,Instagram_Duration,Instagram_Minutes
0,1,2 jam 3 menit,123.0,1 jam 3 menit,63.0
1,2,2 jam 4 menit,124.0,17 menit,17.0
2,3,1 jam 40 menit,100.0,2 jam,120.0
3,4,1 jam,60.0,sepuluh menit,10.0
4,5,2 jam 39 menit,159.0,29 menit,29.0
5,6,1 jam,60.0,30 menit,30.0
6,7,3 jam 1 menit,181.0,1 jam 45 menit,105.0
7,8,3 jam 1 menit,181.0,1 jam,60.0
8,9,2 jam,120.0,1 jam,60.0
9,10,30 menit,30.0,1 jam,60.0


In [29]:
# 12. Audit Konversi Durasi

tiktok_unparsed = df[
    df['TikTok_Duration'].notna() &
    df['TikTok_Minutes'].isna()
][['id', 'TikTok_Duration']]

instagram_unparsed = df[
    df['Instagram_Duration'].notna() &
    df['Instagram_Minutes'].isna()
][['id', 'Instagram_Duration']]

print("Durasi TikTok yang belum terbaca:")
display(tiktok_unparsed)

print("Durasi Instagram yang belum terbaca:")
display(instagram_unparsed)

Durasi TikTok yang belum terbaca:


,id,TikTok_Duration
22,23,3


Durasi Instagram yang belum terbaca:


,id,Instagram_Duration
22,23,1


In [30]:
# 13. Checking Konsistensi Jawaban

# Tidak menggunakan TikTok tetapi mengisi durasi
tiktok_conflict = df[
    (df['TikTok_Status'] == 'Tidak') &
    (
        df['TikTok_Duration'].notna() |
        df['TikTok_Purpose'].notna()
    )
]

# Tidak menggunakan Instagram tetapi mengisi durasi
instagram_conflict = df[
    (df['Instagram_Status'] == 'Tidak') &
    (
        df['Instagram_Duration'].notna() |
        df['Instagram_Purpose'].notna()
    )
]

print("Konflik jawaban TikTok:")
display(tiktok_conflict)

print("Konflik jawaban Instagram:")
display(instagram_conflict)

Konflik jawaban TikTok:


,id,Original_ID,Gender,Major,Year,TikTok_Status,TikTok_Duration,TikTok_Purpose,Instagram_Status,Instagram_Duration,Instagram_Purpose,TikTok_Minutes,Instagram_Minutes


Konflik jawaban Instagram:


,id,Original_ID,Gender,Major,Year,TikTok_Status,TikTok_Duration,TikTok_Purpose,Instagram_Status,Instagram_Duration,Instagram_Purpose,TikTok_Minutes,Instagram_Minutes


In [31]:
# 14. Checking Durasi Tidak Wajar

tiktok_abnormal = df[
    df['TikTok_Minutes'] > 1440
]

instagram_abnormal = df[
    df['Instagram_Minutes'] > 1440
]

print("Durasi TikTok lebih dari 24 jam:")
display(tiktok_abnormal)

print("Durasi Instagram lebih dari 24 jam:")
display(instagram_abnormal)

Durasi TikTok lebih dari 24 jam:


,id,Original_ID,Gender,Major,Year,TikTok_Status,TikTok_Duration,TikTok_Purpose,Instagram_Status,Instagram_Duration,Instagram_Purpose,TikTok_Minutes,Instagram_Minutes


Durasi Instagram lebih dari 24 jam:


,id,Original_ID,Gender,Major,Year,TikTok_Status,TikTok_Duration,TikTok_Purpose,Instagram_Status,Instagram_Duration,Instagram_Purpose,TikTok_Minutes,Instagram_Minutes


In [32]:
# 15. Ringkasan Data Cleaning dan Checking

summary = pd.DataFrame({

    'Pemeriksaan': [
        'Jumlah Responden',
        'Jumlah Variabel',
        'Duplikasi Baris',
        'Duplikasi ID Asli',
        'Total Missing Value',
        'Pengguna TikTok',
        'Pengguna Instagram',
        'Durasi TikTok Belum Terbaca',
        'Durasi Instagram Belum Terbaca'
    ],

    'Hasil': [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df['Original_ID'].duplicated().sum(),
        df.isnull().sum().sum(),
        (df['TikTok_Status'] == 'Ya').sum(),
        (df['Instagram_Status'] == 'Ya').sum(),
        len(tiktok_unparsed),
        len(instagram_unparsed)
    ]

})

display(summary)

,Pemeriksaan,Hasil
0,Jumlah Responden,100
1,Jumlah Variabel,13
2,Duplikasi Baris,0
3,Duplikasi ID Asli,0
4,Total Missing Value,59
5,Pengguna TikTok,84
6,Pengguna Instagram,97
7,Durasi TikTok Belum Terbaca,1
8,Durasi Instagram Belum Terbaca,1


In [33]:
# 16. Export Dataset

output_file = 'Dataset_TikTok_Instagram_Cleaned.csv'

df.to_csv(output_file, index=False)

print("Dataset berhasil disimpan.")

files.download(output_file)

Dataset berhasil disimpan.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>